# 01 — Run cell segmentation

Thin wrapper around the shared `cell_tracker` package (`python pipelines/cell segmentation/ilastik/cell tracker`). This dataset has **no nuclear stain** (macrophages in Green - 488, target readout in Far Red - 640), so segmentation uses the `green` panel -- its own independently trained 3-class ilastik classifier (`membrane` / `background` / `cell_contact`, see `cell tracker/green/README.md`) -- and splits touching cells directly from the membrane mask via distance-transform watershed (`cell_tracker.instance_segmentation.cells_from_membrane_mask`) instead of nucleus-seeded watershed, since there's no nucleus to seed from.

**Test vs full data toggle:** `config.yaml`'s `use_test_data` flag picks between the small always-available test file (`data/test/`) and the full `data/raw/<replicate>/<condition>/` tree -- flip it there, not in this notebook.

**Multi-point files:** each ND2 file can contain multiple points (fields of view) -- `run_segmentation_pipeline` automatically splits every point into its own row in the manifest below and segments each independently.

**QC:** a QC panel (input image next to what was identified -- membrane threshold, final cell instances) is written for the first point processed by default, per `qc_first_only` in `config.yaml`. Set `qc_point_indices` in `config.yaml` (e.g. `[5, 16, 24]`) to instead always get QC for specific points regardless of which happens to be first -- note `point_index` alone matches that index in EVERY file, not just one. This notebook also always generates QC for every replicate's background-subtraction reference point (see `background_subtraction.reference_points` in `config.yaml`), targeted precisely by (replicate, condition, point_index) so it doesn't accidentally match unrelated files that happen to share the same point_index.

**Frame exclusion:** disabled for this pipeline (`max_membrane_area_fraction: null` in the `green` panel's `panel.yaml`). Macrophages legitimately cover most of the frame in this assay, so the 30% membrane-area cutoff that makes sense for Erika TNFa's sparser nuclear-seeded pipeline flagged every single point here -- not a real quality signal. The manifest still carries `membrane_area_fraction` / `excluded` / `exclude_reason` columns for consistency with the shared pipeline, but `excluded` will always read `False` unless you set a threshold back in `panel.yaml`.

**Prerequisite:** the `green` panel needs its own trained `membrane_segmentation.ilp` (see `cell tracker/green/README.md`'s training checklist if you haven't yet).

In [ ]:
from cell_tracker import (
    load_config, load_panel, discover_project_files, discover_flat_files,
    run_segmentation_pipeline,
)

cfg = load_config('../config.yaml')

panels_root = (cfg['_project_root'] / cfg['segmentation']['panels_root']).resolve()
panel = load_panel(panels_root, cfg['segmentation']['panel_name'])
print('Using panel:', panel['panel_name'], '->', panel['ilastik_project_path'])

use_test_data = cfg.get('use_test_data', True)
if use_test_data:
    files_df = discover_flat_files(cfg['paths']['test_dir'])
    print(f'TEST MODE -- running against {len(files_df)} file(s) in data/test/. '
          "Set use_test_data: false in config.yaml for a full run.")
else:
    files_df = discover_project_files(cfg['paths']['raw_dir'])
    print(f'FULL RUN -- {len(files_df)} file(s) found under data/raw/.')
files_df

In [ ]:
import inspect
from cell_tracker import run_segmentation_pipeline
print('qc_point_indices' in inspect.signature(run_segmentation_pipeline).parameters)
print('nuclear_channel' in inspect.signature(run_segmentation_pipeline).parameters)
print('qc_selectors' in inspect.signature(run_segmentation_pipeline).parameters)
# All three must print True. If any prints False, the kernel is running an
# old install of cell_tracker -- reinstall (pip install -e ...) and
# restart the kernel before doing anything else.

In [ ]:
# QC is always generated for each replicate's background-subtraction
# reference point (config.yaml's background_subtraction.reference_points),
# in addition to whatever qc_first_only/qc_point_indices already select --
# these points matter more than most, since an error there skews background
# for every point in that replicate, not just itself.
bg_cfg = cfg.get('background_subtraction', {})
qc_selectors = None
if bg_cfg.get('enabled') and bg_cfg.get('method') == 'segmentation_background':
    qc_selectors = [
        (replicate, ref['condition'], ref['point_index'])
        for replicate, ref in bg_cfg.get('reference_points', {}).items()
    ]
    print(f'Always generating QC for {len(qc_selectors)} background reference point(s): {qc_selectors}')

manifest, manifest_path = run_segmentation_pipeline(
    files_df=files_df,
    panel=panel,
    nuclear_channel=None,  # no nuclear stain in this dataset -- switches to
                            # distance-transform watershed (cells_from_membrane_mask)
    ilastik_executable=cfg['segmentation']['ilastik_executable'],
    work_dir=cfg['paths']['segmentation_work_dir'],
    zstack_cfg=cfg.get('zstack'),
    qc_dir=cfg['paths']['qc_dir'],
    qc_first_only=cfg.get('qc_first_only', True),
    qc_point_indices=cfg.get('qc_point_indices'),
    qc_selectors=qc_selectors,
)

print(f'Segmented {len(manifest)} point(s) across {manifest["file_id"].nunique()} file(s). '
      f'Manifest -> {manifest_path}')
manifest[['replicate', 'condition', 'filename', 'point_label', 'membrane_area_fraction', 'excluded', 'segmentation_path']]

## Cell counts per point

In [ ]:
import numpy as np
from tqdm.auto import tqdm

from cell_tracker import load_segmentation

object_counts = []
for row in tqdm(manifest.itertuples(), total=len(manifest), desc='Counting cells'):
    label_img = load_segmentation(row.segmentation_path)
    object_counts.append(len(np.unique(label_img)) - 1)

manifest['n_cells'] = object_counts
manifest.to_csv(manifest_path, index=False)
manifest[['replicate', 'condition', 'filename', 'point_label', 'n_cells']]

## Excluded frames

Frame exclusion is disabled for this pipeline (see the intro above) -- `max_membrane_area_fraction` is `null` in the `green` panel's `panel.yaml`, so this table should always come back empty. Kept here for consistency with the shared pipeline and in case you re-enable a threshold later.

The `exclude_reason` column can get cut off by pandas' default display width -- the print statement below shows it in full.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

excluded = manifest[manifest['excluded']]
print(f'{len(excluded)} of {len(manifest)} point(s) excluded:')
for row in excluded.itertuples():
    print(f'  {row.point_label}: {row.exclude_reason}')

excluded[['point_label', 'membrane_area_fraction', 'exclude_reason']]

## QC -- inspect pipeline panels

Shows membrane threshold and final cell instances -- each as input image next to what was identified. Generated for the points listed in `qc_point_indices` (config.yaml), the first point if that's not set, and always for every replicate's background-subtraction reference point (see the pipeline-run cell above). Open `data/qc/` directly for more.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

qc_pngs = sorted(cfg['paths']['qc_dir'].glob('*_qc.png'))
print(f'{len(qc_pngs)} QC panel(s) in data/qc/:')
for p in qc_pngs:
    print(' -', p.name)

for p in qc_pngs:
    img = mpimg.imread(p)
    fig, ax = plt.subplots(figsize=(10, 15))
    ax.imshow(img)
    ax.set_title(p.name)
    ax.axis('off')
    plt.show()

## Next step

Confirm the QC panel above looks right (membrane threshold capturing real cells, final boundaries not badly merged/fragmented, touching cells correctly split at their `cell_contact` boundary) -- especially the background reference point QC panels, since an error there skews background for every point in that replicate. If not, tune `panel.yaml`'s `segmentation:` block (see `cell tracker/green/README.md`'s "Tuning" section) -- `smoothing_sigma` / `probability_threshold` / `min_cell_size_px` / `min_seed_distance_px` / `contact_probability_threshold` -- then re-run this notebook with `force_segment=True` passed to `run_segmentation_pipeline`. `fill_holes` should stay off for this panel (see README -- it's incompatible with `contact_class`).

Frame exclusion is off for this pipeline, so the excluded-frames table above should stay empty -- if you want it back (e.g. to catch genuinely oversaturated/artifact frames), set `max_membrane_area_fraction` in the `green` panel's `panel.yaml` to a value that fits this assay's normal membrane coverage, not the nuclear-pipeline default. Once segmentation looks good, continue to `02_measure_target_intensity.ipynb`. When you're ready for a real run (not just the test file), set `use_test_data: false` in `config.yaml` and re-run this notebook.